# 08 — Change Analysis
**Steps 30–32:** Compute annual landcover area statistics, analyse trends 2018–2025, and generate multi-temporal trend plots.

In [ ]:
import os
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

EXPORTS_DIR = os.path.join('..', 'data', 'exports')
OUTPUTS_DIR = os.path.join('..', 'outputs')

YEARS       = list(range(2018, 2026))
CLASS_NAMES = ['Water', 'Trees', 'Crops', 'Built-up Areas', 'Bare Ground']
CLASS_CODES = [1, 2, 3, 4, 5]
CLASS_COLORS = ['#4472C4', '#70AD47', '#FFC000', '#FF0000', '#A6A6A6']

PIXEL_AREA_M2 = 10 * 10           # 10 m resolution → 100 m²
PIXEL_AREA_KM2 = PIXEL_AREA_M2 / 1e6

In [ ]:
# ── STEP 30: Compute Area Statistics ────────────────────────

def compute_area_stats(year):
    """Count pixels per class and convert to km²."""
    map_path = os.path.join(EXPORTS_DIR, f'landcover_{year}.tif')
    with rasterio.open(map_path) as src:
        lc_map = src.read(1)

    row = {'Year': year}
    for code, name in zip(CLASS_CODES, CLASS_NAMES):
        count = (lc_map == code).sum()
        row[name] = round(count * PIXEL_AREA_KM2, 4)
    return row

area_records = [compute_area_stats(y) for y in YEARS]
area_df = pd.DataFrame(area_records).set_index('Year')

# Add totals column
area_df['Total'] = area_df[CLASS_NAMES].sum(axis=1)

print('Annual Landcover Area (km²):')
print(area_df.to_string())

area_df.to_csv(os.path.join(OUTPUTS_DIR, 'annual_area_stats.csv'))
print('\n✅ Area stats saved to outputs/annual_area_stats.csv')

In [ ]:
# ── STEP 31: Analyse Landcover Trends ────────────────────────

print('LANDCOVER CHANGE SUMMARY (2018 → 2025)')
print('=' * 50)
for name in CLASS_NAMES:
    start = area_df.loc[2018, name]
    end   = area_df.loc[2025, name]
    delta = end - start
    pct   = (delta / start) * 100 if start > 0 else 0
    direction = '▲' if delta > 0 else '▼'
    print(f'  {name:20s}: {start:8.2f} → {end:8.2f} km²  {direction} {abs(delta):.2f} km² ({pct:+.1f}%)')

In [ ]:
# ── STEP 32: Multi-Temporal Trend Plots ──────────────────────

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

# Individual class trend lines
for i, (name, color) in enumerate(zip(CLASS_NAMES, CLASS_COLORS)):
    ax = axes[i]
    ax.plot(YEARS, area_df[name].values, marker='o', linewidth=2.5,
            color=color, markersize=7)
    ax.fill_between(YEARS, area_df[name].values, alpha=0.15, color=color)
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('Area (km²)')
    ax.set_xticks(YEARS)
    ax.tick_params(axis='x', rotation=45)
    ax.grid(alpha=0.3)

# Combined trend on last subplot
ax = axes[5]
for name, color in zip(CLASS_NAMES, CLASS_COLORS):
    ax.plot(YEARS, area_df[name].values, marker='o', linewidth=2,
            label=name, color=color, markersize=5)
ax.set_title('All Classes', fontsize=12, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Area (km²)')
ax.set_xticks(YEARS)
ax.tick_params(axis='x', rotation=45)
ax.legend(fontsize=8, loc='upper left')
ax.grid(alpha=0.3)

plt.suptitle('Landcover Trends — Obuasi Municipality, Ghana (2018–2025)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, 'landcover_trends.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Trend plots saved to outputs/landcover_trends.png')

In [ ]:
# ── Stacked Area Chart ────────────────────────────────────────

fig, ax = plt.subplots(figsize=(12, 6))

areas = [area_df[name].values for name in CLASS_NAMES]
ax.stackplot(YEARS, areas, labels=CLASS_NAMES, colors=CLASS_COLORS, alpha=0.85)

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Area (km²)', fontsize=12)
ax.set_title('Stacked Annual Landcover Area — Obuasi Municipality', fontsize=13, fontweight='bold')
ax.set_xticks(YEARS)
ax.legend(loc='upper left', fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, 'stacked_area_chart.png'), dpi=150)
plt.show()
print('✅ Stacked area chart saved.')